# SA 4 - PCA

## Justificación

Comparo PCA con los datos originales y con los datos escalados. Después conservo el menor número de componentes que alcanza 90% de varianza acumulada.

## Diseño del modelo

El proceso está ordenado en celdas: carga, preparación, ajuste, evaluación y guardado del modelo.

Repositorio: https://github.com/axelisaak10/evaluacion-2

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

ruta = Path("..")
datos = pd.read_csv(ruta / "data" / "comprar_alquilar.csv")
datos.head()

## Preparación

In [ ]:
columnas = [columna for columna in datos.columns if columna != "comprar"]
x = datos[columnas]

pca_sin_escala = PCA()
componentes_sin_escala = pca_sin_escala.fit_transform(x)

escala = StandardScaler()
x_escalado = escala.fit_transform(x)

pca = PCA()
componentes = pca.fit_transform(x_escalado)
varianza = np.cumsum(pca.explained_variance_ratio_)

cantidad = int(np.argmax(varianza >= 0.90) + 1)
cantidad

## Comparación de los primeros componentes

In [ ]:
figura, ejes = plt.subplots(1, 2, figsize=(11, 5))

ejes[0].scatter(
    componentes_sin_escala[:, 0],
    componentes_sin_escala[:, 1],
    c=datos["comprar"],
    cmap="coolwarm",
    alpha=0.75,
)
ejes[0].set_title("Sin escalamiento")

ejes[1].scatter(
    componentes[:, 0],
    componentes[:, 1],
    c=datos["comprar"],
    cmap="coolwarm",
    alpha=0.75,
)
ejes[1].set_title("Con escalamiento")

for grafica in ejes:
    grafica.set_xlabel("PC1")
    grafica.set_ylabel("PC2")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(varianza) + 1), varianza, "o-")
plt.axhline(0.90, linestyle="--", color="red")
plt.axvline(cantidad, linestyle="--", color="green")
plt.xlabel("Número de componentes")
plt.ylabel("Varianza acumulada")
plt.title("Selección de componentes")
plt.show()

In [ ]:
pca_final = PCA(n_components=cantidad)
pca_final.fit(x_escalado)

joblib.dump(
    {
        "columnas": columnas,
        "escala": escala,
        "pca": pca_final,
        "componentes": cantidad,
    },
    ruta / "models" / "sa_pca_comprar_alquilar.joblib",
)

## Interpretación

Sin escalamiento dominan las variables con cantidades grandes. Con escalamiento las variables quedan en condiciones comparables. Se eligieron seis componentes porque son los primeros que superan 90% de varianza acumulada.